In [1]:
import os
import sys
import time
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from typing import Literal

C:\Users\srush\AppData\Local\Temp\ipykernel_15432\2498989732.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


faiss_path = (
    PROJECT_ROOT
    / "Data"
    / "Cleaned"
    / "faiss_index"
)

vector_db = FAISS.load_local(
    folder_path=str(faiss_path),
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)


reviews_path = (
    PROJECT_ROOT
    / "Data"
    / "Cleaned"
    / "reviews_sample.parquet"
)

reviews_df = pd.read_parquet(
    reviews_path
)

print("Single-agent resources loaded successfully.")

Single-agent resources loaded successfully.


In [3]:
class SingleAgentProduct(BaseModel):
    rank: int
    parent_asin: str
    title: str
    reason: str


class SingleAgentOutput(BaseModel):
    products: list[SingleAgentProduct] = Field(
        default_factory=list
    )

    recommended_product_asin: str | None = None
    recommended_product_title: str | None = None

    confidence: Literal[
    "high",
    "medium",
    "low"
]
    summary: str

In [4]:
def retrieve_single_agent_candidates(
    user_query: str,
    k: int = 10
) -> list[dict]:

    results = (
        vector_db.similarity_search_with_score(
            user_query,
            k=k
        )
    )

    candidates = []

    for document, distance in results:

        metadata = document.metadata.copy()

        candidates.append(
            {
                "parent_asin": str(
                    metadata.get(
                        "parent_asin",
                        ""
                    )
                ),

                "title": str(
                    metadata.get(
                        "title",
                        ""
                    )
                ),

                "price": metadata.get(
                    "price"
                ),

                "average_rating": metadata.get(
                    "average_rating",
                    0
                ),

                "rating_number": metadata.get(
                    "rating_number",
                    0
                ),

                "categories": metadata.get(
                    "categories",
                    []
                ),

                "product_text": document.page_content,

                "retrieval_distance": float(
                    distance
                )
            }
        )

    return candidates

In [5]:
def get_review_evidence(
    parent_asin: str,
    max_reviews: int = 3
) -> dict:

    product_reviews = reviews_df[
        reviews_df["parent_asin"]
        == parent_asin
    ]

    if product_reviews.empty:
        return {
            "review_count": 0,
            "average_review_rating": 0,
            "reviews": []
        }

    reviews = (
        product_reviews["text"]
        .dropna()
        .head(max_reviews)
        .astype(str)
        .tolist()
    )

    return {
        "review_count": len(
            product_reviews
        ),

        "average_review_rating": round(
            float(
                product_reviews["rating"]
                .mean()
            ),
            2
        ),

        "reviews": reviews
    }

In [6]:
def enrich_single_agent_candidates(
    candidates: list[dict],
    max_reviews: int = 3
) -> list[dict]:

    enriched = []

    for candidate in candidates:

        review_evidence = (
            get_review_evidence(
                parent_asin=(
                    candidate["parent_asin"]
                ),
                max_reviews=max_reviews
            )
        )

        enriched.append(
            {
                **candidate,
                **review_evidence
            }
        )

    return enriched

In [7]:
class SingleRecommendationAgent:

    def __init__(
        self,
        llm: ChatOpenAI
    ):

        self.structured_llm = (
            llm.with_structured_output(
                SingleAgentOutput
            )
        )

    def recommend(
        self,
        user_query: str,
        candidates: list[dict]
    ) -> SingleAgentOutput:

        compact_candidates = []

        for candidate in candidates:

            compact_candidates.append(
                {
                    "parent_asin": (
                        candidate[
                            "parent_asin"
                        ]
                    ),

                    "title": (
                        candidate["title"]
                    ),

                    "price": (
                        candidate.get(
                            "price"
                        )
                    ),

                    "average_rating": (
                        candidate.get(
                            "average_rating",
                            0
                        )
                    ),

                    "rating_number": (
                        candidate.get(
                            "rating_number",
                            0
                        )
                    ),

                    "product_text": str(
                        candidate.get(
                            "product_text",
                            ""
                        )
                    )[:500],

                    "review_count": (
                        candidate.get(
                            "review_count",
                            0
                        )
                    ),

                    "reviews": [
                        str(review)[:250]
                        for review in candidate.get(
                            "reviews",
                            []
                        )[:3]
                    ]
                }
            )

        prompt = f"""
You are a single-agent ecommerce recommendation system.

You must perform all tasks yourself:

1. Understand the customer request.
2. Check product type and brand.
3. Check budget when one is provided.
4. Evaluate requested features.
5. Use product metadata and customer reviews.
6. Reject accessories when the customer requested a main product.
7. Rank the valid products.
8. Select the best recommendation.

Customer query:

{user_query}

Retrieved candidates:

{json.dumps(
    compact_candidates,
    default=str
)}

Rules:

- Use only the supplied candidate evidence.
- Do not use outside product knowledge.
- Do not invent specifications.
- Do not recommend accessories as phones, laptops, headphones,
  earbuds or other main products.
- When a budget is stated, do not recommend a product with an
  unavailable or over-budget price.
- Missing reviews indicate limited evidence, not poor quality.
- Return no recommendation if no candidate satisfies the main
  product type or brand requirement.
- Rank at most five products.
- Keep reasons concise.
- Confidence must be exactly one of:
  high, medium, low.
"""

        return self.structured_llm.invoke(
            prompt
        )

In [8]:
single_agent = SingleRecommendationAgent(
    llm=llm
)

In [9]:
def run_single_agent_system(
    user_query: str,
    retrieval_k: int = 10,
    max_reviews: int = 3,
    verbose: bool = True
) -> dict:

    if not user_query or not user_query.strip():
        raise ValueError(
            "User query cannot be empty."
        )

    start_time = time.perf_counter()

    retrieval_start = time.perf_counter()

    candidates = retrieve_single_agent_candidates(
        user_query=user_query,
        k=retrieval_k
    )

    retrieval_latency = (
        time.perf_counter()
        - retrieval_start
    )

    enrichment_start = time.perf_counter()

    enriched_candidates = (
        enrich_single_agent_candidates(
            candidates=candidates,
            max_reviews=max_reviews
        )
    )

    enrichment_latency = (
        time.perf_counter()
        - enrichment_start
    )

    agent_start = time.perf_counter()

    output = single_agent.recommend(
        user_query=user_query,
        candidates=enriched_candidates
    )

    agent_latency = (
        time.perf_counter()
        - agent_start
    )

    total_latency = (
        time.perf_counter()
        - start_time
    )

    if verbose:

        print("=" * 100)
        print("SINGLE-AGENT SYSTEM")
        print("=" * 100)

        print("User query:")
        print(user_query)
        print()

        print(
            "Retrieved candidates:",
            len(candidates)
        )

        print()

        for product in output.products:

            print(
                f"{product.rank}. "
                f"{product.title}"
            )

            print(
                "Reason:",
                product.reason
            )

            print()

        print(
            "Recommended product:",
            output.recommended_product_title
        )

        print(
            "Confidence:",
            output.confidence
        )

        print(
            "Summary:",
            output.summary
        )

        print(
            "Total latency:",
            round(
                total_latency,
                2
            ),
            "seconds"
        )

    return {
        "system": "single_agent",
        "user_query": user_query,
        "candidates": candidates,
        "enriched_candidates": (
            enriched_candidates
        ),
        "output": output,
        "latency": {
            "retrieval": round(
                retrieval_latency,
                4
            ),
            "review_enrichment": round(
                enrichment_latency,
                4
            ),
            "single_agent": round(
                agent_latency,
                4
            ),
            "total": round(
                total_latency,
                4
            )
        }
    }

In [10]:
single_result = run_single_agent_system(
    user_query=(
        "Recommend a Samsung phone with a good "
        "camera and long battery life."
    ),
    retrieval_k=10,
    max_reviews=3,
    verbose=True
)

SINGLE-AGENT SYSTEM
User query:
Recommend a Samsung phone with a good camera and long battery life.

Retrieved candidates: 10

1. Samsung Galaxy S22 Ultra 256GB Unlocked ATT TMobile Verizon 100% batt! 7548400
Reason: High average rating (4.7) and good camera features.

2. SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Black - Unlocked (Renewed Premium)
Reason: High average rating (4.2) and excellent camera capabilities.

3. SAMSUNG Galaxy S21 FE 5G SM-G990U 256GB Factory Unlocked Smartphone (Renewed) (Olive)
Reason: Good average rating (4.7) and features a triple-lens camera.

4. Samsung Galaxy S22 Smartphone, Factory Unlocked Android Cell Phone, 256GB, 8K Camera & Video, Brightest Display, Long Battery Life, Fast 4nm Processor, US Version, Phantom White (Renewed)
Reason: Good average rating (4.1) and strong camera performance.

5. Samsung Galaxy A52 (5G) 128GB A526U 6.5" Display Quad Camera Smartphone - Black (Renewed) (AT&T Unlocked)
Reason: Good average rating (4.2) and decent camera f

In [11]:
# =====================================================
# FINAL SINGLE-AGENT RUN
# 60 QUERIES + TOKENS + LATENCY
# =====================================================

from langchain_community.callbacks.manager import (
    get_openai_callback
)

import pandas as pd


FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)


benchmark_df = pd.read_csv(
    FINAL_FOLDER
    / "final_60_query_benchmark.csv"
)


single_final_rows = []


for _, test_case in benchmark_df.iterrows():

    test_id = int(
        test_case["test_id"]
    )

    user_query = str(
        test_case["user_query"]
    )

    print(
        "Single Agent:",
        test_id
    )


    try:

        with get_openai_callback() as cb:

            result = run_single_agent_system(
                user_query=user_query,
                retrieval_k=10,
                max_reviews=3,
                verbose=False
            )


        output = result[
            "output"
        ]


        top_5_asins = [
            product.parent_asin
            for product
            in output.products[:5]
            if product.parent_asin
        ]


        top_5_titles = [
            product.title
            for product
            in output.products[:5]
            if product.title
        ]


        single_final_rows.append({

            "test_id":
                test_id,

            "category":
                test_case["category"],

            "user_query":
                user_query,

            "recommended_product":
                output.recommended_product_title,

            "recommended_asin":
                output.recommended_product_asin,

            "confidence":
                output.confidence,

            "top_5_asins":
                "|".join(
                    top_5_asins
                ),

            "top_5_titles":
                " || ".join(
                    top_5_titles
                ),

            "total_latency":
                result[
                    "latency"
                ][
                    "total"
                ],

            "prompt_tokens":
                int(
                    cb.prompt_tokens
                ),

            "completion_tokens":
                int(
                    cb.completion_tokens
                ),

            "total_tokens":
                int(
                    cb.total_tokens
                ),

            "estimated_cost_usd":
                float(
                    cb.total_cost
                ),

            "error":
                None
        })


    except Exception as error:

        single_final_rows.append({

            "test_id":
                test_id,

            "category":
                test_case["category"],

            "user_query":
                user_query,

            "recommended_product":
                None,

            "recommended_asin":
                None,

            "confidence":
                "low",

            "top_5_asins":
                "",

            "top_5_titles":
                "",

            "total_latency":
                0,

            "prompt_tokens":
                0,

            "completion_tokens":
                0,

            "total_tokens":
                0,

            "estimated_cost_usd":
                0,

            "error":
                str(error)
        })


single_final_df = pd.DataFrame(
    single_final_rows
)


display(
    single_final_df
)


single_final_df.to_csv(
    FINAL_FOLDER
    / "final_single_agent_results.csv",
    index=False
)


print(
    "Final Single Agent complete."
)

Single Agent: 1
Single Agent: 2
Single Agent: 3
Single Agent: 4
Single Agent: 5
Single Agent: 6
Single Agent: 7
Single Agent: 8
Single Agent: 9
Single Agent: 10
Single Agent: 11
Single Agent: 12
Single Agent: 13
Single Agent: 14
Single Agent: 15
Single Agent: 16
Single Agent: 17
Single Agent: 18
Single Agent: 19
Single Agent: 20
Single Agent: 21
Single Agent: 22
Single Agent: 23
Single Agent: 24
Single Agent: 25
Single Agent: 26
Single Agent: 27
Single Agent: 28
Single Agent: 29
Single Agent: 30
Single Agent: 31
Single Agent: 32
Single Agent: 33
Single Agent: 34
Single Agent: 35
Single Agent: 36
Single Agent: 37
Single Agent: 38
Single Agent: 39
Single Agent: 40
Single Agent: 41
Single Agent: 42
Single Agent: 43
Single Agent: 44
Single Agent: 45
Single Agent: 46
Single Agent: 47
Single Agent: 48
Single Agent: 49
Single Agent: 50
Single Agent: 51
Single Agent: 52
Single Agent: 53
Single Agent: 54
Single Agent: 55
Single Agent: 56
Single Agent: 57
Single Agent: 58
Single Agent: 59
Single

,test_id,category,user_query,recommended_product,recommended_asin,confidence,top_5_asins,top_5_titles,total_latency,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd,error
0,1,Brand Only,Recommend a Samsung phone.,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",B09C6N8P6Y,medium,B09C6N8P6Y|B09S6VKCLX|B0BF1FZTLQ|B09WT8N5X7|B0...,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",8.4858,3128,408,3536,0.000714,None
1,2,Brand Only,Recommend an Apple phone.,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY,medium,B00CX0OZHY|B00BUYRQG6|B07DWFTVKL|B00EELONEU|B0...,Apple iPhone 5 - 16GB (Black) Factory Unlocked...,5.9017,2821,314,3135,0.000612,None
2,3,Brand Only,Suggest a Motorola smartphone.,Motorola DROID A855 Android Phone (Verizon Wir...,B002VRO83K,medium,B002VRO83K|B004AZ4FHA|B006P82YC8|B004P551BE|B0...,Motorola DROID A855 Android Phone (Verizon Wir...,4.7732,2451,322,2773,0.000561,None
3,4,Brand Only,Recommend a Nokia phone.,Nokia E5-00 Unlocked GSM Phone with Easy Email...,B003X26SLM,high,B003X26SLM|B00BIQXW1O|B00P5AVL0I|B07VP8P1Z1|B0...,Nokia E5-00 Unlocked GSM Phone with Easy Email...,6.3362,2743,394,3137,0.000648,None
4,5,Brand Only,Suggest a Google phone.,Google Pixel 6a Phone - Charcoal Pixel 6a Case...,B0B6PV17MC,high,B0B6PV17MC|B09NP4C1MF|B08BXBT8MD|B01M27MVQI|B0...,Google Pixel 6a Phone - Charcoal Pixel 6a Case...,5.3562,2793,335,3128,0.000620,None
5,6,Brand + Feature,Recommend a Samsung phone with a good camera.,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",B09S6VKCLX,high,B09S6VKCLX|B09WT8N5X7|B08Y973HW1|B07QH32PCY,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",4.9867,3089,338,3427,0.000666,None
6,7,Brand + Feature,Recommend a Samsung phone with long battery life.,"Samsung Galaxy S22 Smartphone, Factory Unlocke...",B09WT8N5X7,high,B09WT8N5X7|B09S6VKCLX,"Samsung Galaxy S22 Smartphone, Factory Unlocke...",4.1569,3097,221,3318,0.000597,None
7,8,Brand + Feature,Recommend an Apple phone with a good camera.,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,B078WZX9LD,medium,B078WZX9LD|B00CX0OZHY,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,2.6330,3010,157,3167,0.000546,None
8,9,Brand + Feature,Suggest a Motorola phone with a good camera.,Motorola Verizon Motorola Droid Razr M 4G Smar...,B004P551BE,high,B004P551BE|B002VRO83K|B006P82YC8|B004AZ4FHA|B0...,Motorola Verizon Motorola Droid Razr M 4G Smar...,5.9571,2555,351,2906,0.000594,None
9,10,Brand + Feature,Recommend a Samsung phone with fast performance.,"Samsung Galaxy S22 Smartphone, Factory Unlocke...",B09WT8N5X7,high,B09WT8N5X7|B09S6VKCLX|B0BF1FZTLQ|B09X9FC88M|B0...,"Samsung Galaxy S22 Smartphone, Factory Unlocke...",5.8461,2986,430,3416,0.000706,None


Final Single Agent complete.


In [11]:
from langchain_community.callbacks.manager import get_openai_callback
import pandas as pd

# Load the same 15 test queries
test_queries_df = pd.read_csv(
    PROJECT_ROOT
    / "Results"
    / "coordinated_pipeline_test_results.csv"
)[["test_id", "user_query"]].drop_duplicates(
    subset=["test_id"]
).sort_values("test_id")


single_token_rows = []

print("Queries loaded:", len(test_queries_df))


for _, row in test_queries_df.iterrows():

    test_id = int(row["test_id"])
    user_query = str(row["user_query"])

    print("Running Single Agent:", test_id)

    with get_openai_callback() as cb:

        result = run_single_agent_system(
            user_query=user_query,
            retrieval_k=10,
            max_reviews=3,
            verbose=False
        )

    single_token_rows.append({
        "test_id": test_id,
        "system": "Single Agent",
        "user_query": user_query,
        "prompt_tokens": int(cb.prompt_tokens),
        "completion_tokens": int(cb.completion_tokens),
        "total_tokens": int(cb.total_tokens),
        "estimated_cost_usd": float(cb.total_cost)
    })


single_token_df = pd.DataFrame(single_token_rows)

display(single_token_df)


single_token_df.to_csv(
    PROJECT_ROOT
    / "Results"
    / "single_agent_token_usage.csv",
    index=False
)

print("Saved Single Agent token results.")

Queries loaded: 15
Running Single Agent: 1
Running Single Agent: 2
Running Single Agent: 3
Running Single Agent: 4
Running Single Agent: 5
Running Single Agent: 6
Running Single Agent: 7
Running Single Agent: 8
Running Single Agent: 9
Running Single Agent: 10
Running Single Agent: 11
Running Single Agent: 12
Running Single Agent: 13
Running Single Agent: 14
Running Single Agent: 15


,test_id,system,user_query,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd
0,1,Single Agent,Recommend a Samsung phone with a good camera a...,3014,418,3432,0.000703
1,2,Single Agent,I want an Apple phone with excellent camera qu...,2939,212,3151,0.000568
2,3,Single Agent,Suggest a Google phone with fast performance.,2981,285,3266,0.000618
3,4,Single Agent,Recommend a Samsung phone under 300 with a goo...,3112,237,3349,0.000609
4,5,Single Agent,Recommend a smartphone under 250 with long bat...,3369,497,3866,0.000804
5,6,Single Agent,Recommend a phone with a great camera.,3015,220,3235,0.000584
6,7,Single Agent,Recommend a phone with long battery life.,3432,267,3699,0.000675
7,8,Single Agent,Suggest a smartphone for gaming.,2971,301,3272,0.000626
8,9,Single Agent,"Recommend a Samsung phone with a good camera, ...",3110,419,3529,0.000718
9,10,Single Agent,Recommend an Apple phone with a good camera an...,3008,277,3285,0.000617


Saved Single Agent token results.


### TESTING

In [11]:
# =====================================================
# TEST QUERIES FOR SINGLE-AGENT PIPELINE
# =====================================================

test_queries = [

    # -----------------------------------------
    # Brand + Feature Queries
    # -----------------------------------------

    {
        "test_id": 1,
        "category": "Brand + Features",
        "query": (
            "Recommend a Samsung phone with a good camera "
            "and long battery life."
        )
    },

    {
        "test_id": 2,
        "category": "Brand + Features",
        "query": (
            "I want an Apple phone with excellent camera quality."
        )
    },

    {
        "test_id": 3,
        "category": "Brand + Features",
        "query": (
            "Suggest a Google phone with fast performance."
        )
    },

    # -----------------------------------------
    # Budget Queries
    # -----------------------------------------

    {
        "test_id": 4,
        "category": "Budget",
        "query": (
            "Recommend a Samsung phone under 300 "
            "with a good camera."
        )
    },

    {
        "test_id": 5,
        "category": "Budget",
        "query": (
            "Recommend a smartphone under 250 "
            "with long battery life."
        )
    },

    # -----------------------------------------
    # Feature Only Queries
    # -----------------------------------------

    {
        "test_id": 6,
        "category": "Feature Only",
        "query": (
            "Recommend a phone with a great camera."
        )
    },

    {
        "test_id": 7,
        "category": "Feature Only",
        "query": (
            "Recommend a phone with long battery life."
        )
    },

    {
        "test_id": 8,
        "category": "Feature Only",
        "query": (
            "Suggest a smartphone for gaming."
        )
    },

    # -----------------------------------------
    # Multiple Feature Queries
    # -----------------------------------------

    {
        "test_id": 9,
        "category": "Multiple Features",
        "query": (
            "Recommend a Samsung phone with a good camera, "
            "long battery life and fast performance."
        )
    },

    {
        "test_id": 10,
        "category": "Multiple Features",
        "query": (
            "Recommend an Apple phone with a good camera "
            "and large storage."
        )
    },

    # -----------------------------------------
    # Ambiguous Queries
    # -----------------------------------------

    {
        "test_id": 11,
        "category": "Ambiguous",
        "query": (
            "Recommend the best Samsung phone."
        )
    },

    {
        "test_id": 12,
        "category": "Ambiguous",
        "query": (
            "I need something good for photography."
        )
    },

    # -----------------------------------------
    # Negative Queries
    # -----------------------------------------

    {
        "test_id": 13,
        "category": "Negative",
        "query": (
            "Recommend an Apple phone with stylus support."
        )
    },

    {
        "test_id": 14,
        "category": "Negative",
        "query": (
            "Recommend a Samsung phone with a removable battery "
            "and an excellent camera."
        )
    },

    # -----------------------------------------
    # No Match Query
    # -----------------------------------------

    {
        "test_id": 15,
        "category": "No Match",
        "query": (
            "Recommend a Nokia phone with an 8K camera "
            "and 1TB storage."
        )
    }

]

In [12]:
test_case = test_queries[0]

In [13]:
result = run_single_agent_system(
    user_query=test_case["query"],
    retrieval_k=10,
    max_reviews=3,
    verbose=False
)

In [14]:
# =====================================================
# RUN ALL TEST QUERIES - SINGLE AGENT
# =====================================================

import time
import pandas as pd

test_results = []

for test_case in test_queries:

    print("=" * 100)
    print(
        f"TEST {test_case['test_id']} | {test_case['category']}"
    )
    print("=" * 100)
    print(test_case["query"])
    print()

    try:

        start_time = time.perf_counter()

        result = run_single_agent_system(
            user_query=test_case["query"],
            retrieval_k=10,
            max_reviews=3,
            verbose=False
        )

        execution_time = (
            time.perf_counter()
            - start_time
        )

        output = result["output"]

        top_5_asins = [
                product.parent_asin
                for product in output.products[:5]
                if product.parent_asin
            ]

        top_5_titles = [
                product.title
                for product in output.products[:5]
                if product.title
            ]

        test_results.append({

            "test_id": test_case["test_id"],

            "category": test_case["category"],

            "user_query": test_case["query"],

            "retrieved_count": len(
                result["candidates"]
            ),

            "ranked_count": len(
                output.products
            ),

            "recommended_product":
                output.recommended_product_title,

            "recommended_asin":
                output.recommended_product_asin,

            "confidence":
                output.confidence,

            "summary":
                output.summary,

            "retrieval_latency":
                result["latency"]["retrieval"],

            "review_latency":
                result["latency"]["review_enrichment"],

            "single_agent_latency":
                result["latency"]["single_agent"],

            "total_latency":
                result["latency"]["total"],

            "top_5_asins": "|".join(top_5_asins),

            "top_5_titles": " || ".join(top_5_titles),

            "error": None

        })

        print(
            "Recommended:",
            output.recommended_product_title
        )

        print(
            "Confidence:",
            output.confidence
        )

        print(
            "Latency:",
            round(execution_time,2),
            "seconds"
        )

    except Exception as error:

        print(
            "Test failed:",
            type(error).__name__,
            str(error)
        )

        test_results.append({

            "test_id": test_case["test_id"],

            "category": test_case["category"],

            "user_query": test_case["query"],

            "retrieved_count":0,

            "ranked_count":0,

            "recommended_product":None,

            "recommended_asin":None,

            "confidence":"low",

            "summary":None,

            "retrieval_latency":0,

            "review_latency":0,

            "single_agent_latency":0,

            "total_latency":0,

            "top_5_asins": "",
            
            "top_5_titles": "",

            "error":str(error)

        })

TEST 1 | Brand + Features
Recommend a Samsung phone with a good camera and long battery life.

Recommended: Samsung Galaxy S22 Ultra 256GB Unlocked ATT TMobile Verizon 100% batt! 7548400
Confidence: high
Latency: 6.49 seconds
TEST 2 | Brand + Features
I want an Apple phone with excellent camera quality.

Recommended: Apple iPhone 7 - 256GB - GSM Unlocked - Red (Renewed)
Confidence: medium
Latency: 3.41 seconds
TEST 3 | Brand + Features
Suggest a Google phone with fast performance.

Recommended: Google Pixel 4a 5G UW Just Black-Verizon (Renewed)
Confidence: high
Latency: 5.24 seconds
TEST 4 | Budget
Recommend a Samsung phone under 300 with a good camera.

Recommended: SAMSUNG Galaxy S9 G960U 64GB Unlocked GSM 4G LTE Phone w/ 12MP Camera - Midnight Black
Confidence: high
Latency: 4.38 seconds
TEST 5 | Budget
Recommend a smartphone under 250 with long battery life.

Recommended: OUKITEL 8000mAh Large Battery 18W Flash Charge Outdoor Phone Android 10 4G 6+128GB Dual SIM Smartphone WP7 IP68

In [15]:
# =====================================================
# CREATE RESULTS DATAFRAME
# =====================================================

single_agent_results_df = pd.DataFrame(
    test_results
)

display(
    single_agent_results_df
)

,test_id,category,user_query,retrieved_count,ranked_count,recommended_product,recommended_asin,confidence,summary,retrieval_latency,review_latency,single_agent_latency,total_latency,top_5_asins,top_5_titles,error
0,1,Brand + Features,Recommend a Samsung phone with a good camera a...,10,5,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,B09X9FC88M,high,The Samsung Galaxy S22 Ultra is the best recom...,0.8235,0.0498,5.6213,6.4946,B09X9FC88M|B09S6VKCLX|B0BF1FZTLQ|B09WT8N5X7|B0...,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,None
1,2,Brand + Features,I want an Apple phone with excellent camera qu...,10,3,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,B078WZX9LD,medium,"The best recommendation is the Apple iPhone 7,...",0.3757,0.0953,2.9352,3.4062,B078WZX9LD|B00CX0OZHY|B00BUYRQG6,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,None
2,3,Brand + Features,Suggest a Google phone with fast performance.,10,3,Google Pixel 4a 5G UW Just Black-Verizon (Rene...,B09NP4C1MF,high,The Google Pixel 4a 5G offers fast performance...,0.2835,0.0836,4.8763,5.2435,B09NP4C1MF|B08BXBT8MD|B01M27MVQI,Google Pixel 4a 5G UW Just Black-Verizon (Rene...,None
3,4,Budget,Recommend a Samsung phone under 300 with a goo...,10,3,SAMSUNG Galaxy S9 G960U 64GB Unlocked GSM 4G L...,B07QH32PCY,high,The Samsung Galaxy S9 is the best recommendati...,0.3885,0.0864,3.9052,4.3800,B07QH32PCY|B00LMJDP1Y|B00CGIULGC,SAMSUNG Galaxy S9 G960U 64GB Unlocked GSM 4G L...,None
4,5,Budget,Recommend a smartphone under 250 with long bat...,10,5,OUKITEL 8000mAh Large Battery 18W Flash Charge...,B0995SX8X8,high,Recommended the OUKITEL smartphone for its hig...,0.4487,0.1030,6.0048,6.5566,B0995SX8X8|B01LZ8516T|B07B944RHF|B07RB4ZH83|B0...,OUKITEL 8000mAh Large Battery 18W Flash Charge...,None
5,6,Feature Only,Recommend a phone with a great camera.,10,5,"ASUS ZenFone 3 MAX ZC520TL Smartphone, 5.2-inc...",B01LZ8516T,medium,The ASUS ZenFone 3 MAX is recommended for its ...,0.5903,0.0853,4.8483,5.5239,B01LZ8516T|B07QYTKZMT|B07F6ZM9QJ|B00ZAJB9C4|B0...,"ASUS ZenFone 3 MAX ZC520TL Smartphone, 5.2-inc...",None
6,7,Feature Only,Recommend a phone with long battery life.,10,2,OUKITEL 8000mAh Large Battery 18W Flash Charge...,B0995SX8X8,high,Recommended the OUKITEL phone for its exceptio...,0.4400,0.0922,3.5375,4.0696,B0995SX8X8|B01LZ8516T,OUKITEL 8000mAh Large Battery 18W Flash Charge...,None
7,8,Feature Only,Suggest a smartphone for gaming.,10,2,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",B09L4QQCN2,high,The Black Shark 4 is the best option for gamin...,0.4289,0.0952,3.5065,4.0306,B09L4QQCN2|B07F4BSXMQ,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",None
8,9,Multiple Features,"Recommend a Samsung phone with a good camera, ...",10,5,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,B09X9FC88M,high,The Samsung Galaxy S22 Ultra is the best recom...,0.2837,0.1000,4.7208,5.1046,B09X9FC88M|B09S6VKCLX|B0BF1FZTLQ|B0B1QXHB7L|B0...,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,None
9,10,Multiple Features,Recommend an Apple phone with a good camera an...,10,4,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,B078WZX9LD,high,The best recommendation is the Apple iPhone 7 ...,0.3220,0.0512,3.0051,3.3782,B078WZX9LD|B00CX0OZHY|B00BUYRQG6|B07DWFTVKL,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,None


In [16]:
# =====================================================
# DISPLAY SUMMARY TABLE
# =====================================================

summary_columns = [
    "test_id",
    "category",
    "user_query",

    "retrieved_count",
    "ranked_count",

    "recommended_product",
    "recommended_asin",

    "top_5_asins",
    "top_5_titles",

    "confidence",

    "total_latency",

    "error"
]

display(
    single_agent_results_df[
        summary_columns
    ]
)

,test_id,category,user_query,retrieved_count,ranked_count,recommended_product,recommended_asin,top_5_asins,top_5_titles,confidence,total_latency,error
0,1,Brand + Features,Recommend a Samsung phone with a good camera a...,10,5,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,B09X9FC88M,B09X9FC88M|B09S6VKCLX|B0BF1FZTLQ|B09WT8N5X7|B0...,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,high,6.4946,None
1,2,Brand + Features,I want an Apple phone with excellent camera qu...,10,3,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,B078WZX9LD,B078WZX9LD|B00CX0OZHY|B00BUYRQG6,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,medium,3.4062,None
2,3,Brand + Features,Suggest a Google phone with fast performance.,10,3,Google Pixel 4a 5G UW Just Black-Verizon (Rene...,B09NP4C1MF,B09NP4C1MF|B08BXBT8MD|B01M27MVQI,Google Pixel 4a 5G UW Just Black-Verizon (Rene...,high,5.2435,None
3,4,Budget,Recommend a Samsung phone under 300 with a goo...,10,3,SAMSUNG Galaxy S9 G960U 64GB Unlocked GSM 4G L...,B07QH32PCY,B07QH32PCY|B00LMJDP1Y|B00CGIULGC,SAMSUNG Galaxy S9 G960U 64GB Unlocked GSM 4G L...,high,4.3800,None
4,5,Budget,Recommend a smartphone under 250 with long bat...,10,5,OUKITEL 8000mAh Large Battery 18W Flash Charge...,B0995SX8X8,B0995SX8X8|B01LZ8516T|B07B944RHF|B07RB4ZH83|B0...,OUKITEL 8000mAh Large Battery 18W Flash Charge...,high,6.5566,None
5,6,Feature Only,Recommend a phone with a great camera.,10,5,"ASUS ZenFone 3 MAX ZC520TL Smartphone, 5.2-inc...",B01LZ8516T,B01LZ8516T|B07QYTKZMT|B07F6ZM9QJ|B00ZAJB9C4|B0...,"ASUS ZenFone 3 MAX ZC520TL Smartphone, 5.2-inc...",medium,5.5239,None
6,7,Feature Only,Recommend a phone with long battery life.,10,2,OUKITEL 8000mAh Large Battery 18W Flash Charge...,B0995SX8X8,B0995SX8X8|B01LZ8516T,OUKITEL 8000mAh Large Battery 18W Flash Charge...,high,4.0696,None
7,8,Feature Only,Suggest a smartphone for gaming.,10,2,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",B09L4QQCN2,B09L4QQCN2|B07F4BSXMQ,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",high,4.0306,None
8,9,Multiple Features,"Recommend a Samsung phone with a good camera, ...",10,5,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,B09X9FC88M,B09X9FC88M|B09S6VKCLX|B0BF1FZTLQ|B0B1QXHB7L|B0...,Samsung Galaxy S22 Ultra 256GB Unlocked ATT TM...,high,5.1046,None
9,10,Multiple Features,Recommend an Apple phone with a good camera an...,10,4,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,B078WZX9LD,B078WZX9LD|B00CX0OZHY|B00BUYRQG6|B07DWFTVKL,Apple iPhone 7 - 256GB - GSM Unlocked - Red (R...,high,3.3782,None


In [17]:
# =====================================================
# SAVE SINGLE-AGENT RESULTS
# =====================================================

from pathlib import Path

results_folder = (
    PROJECT_ROOT
    / "Results" 
)

results_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    results_folder
    / "single_agent_test_results.csv"
)

single_agent_results_df.to_csv(
    output_path,
    index=False
)

print(
    "Single-agent results saved to:",
    output_path
)

Single-agent results saved to: C:\Users\srush\Desktop\Multi agent coordination\Results\single_agent_test_results.csv


In [18]:
# =====================================================
# BASIC LATENCY SUMMARY
# =====================================================

successful_results = single_agent_results_df[
    single_agent_results_df["error"].isna()
]

print(
    "Successful tests:",
    len(successful_results)
)

print(
    "Failed tests:",
    single_agent_results_df["error"]
    .notna()
    .sum()
)

print(
    "Average total latency:",
    round(
        successful_results[
            "total_latency"
        ].mean(),
        4
    ),
    "seconds"
)

print(
    "Median total latency:",
    round(
        successful_results[
            "total_latency"
        ].median(),
        4
    ),
    "seconds"
)

Successful tests: 15
Failed tests: 0
Average total latency: 4.7639 seconds
Median total latency: 5.0078 seconds
